<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6D_Cell_6D_1B0_Reference_Authenticity_Claim_Support_Audit_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GES Stage 6D — Cell 6D-1B0
## Reference Authenticity, DOI/PMID, Retraction/Correction, and Claim-Support Audit



This notebook audits the **17 references in the strengthened GES temporal-validation/RAG manuscript** against Crossref and PubMed, flags DOI/title mismatches, searches for corrected identifiers when needed, screens PubMed correction/retraction links, and materializes a manual claim-support checklist.

### Important boundary
Automated metadata matching can verify that a cited work exists and whether its bibliographic metadata are consistent. It **cannot determine by itself that a paper actually supports a specific manuscript claim**. The final claim-support column must be manually confirmed from the source text before submission.

This notebook deliberately keeps:
- the originally entered DOI/title,
- the independently retrieved Crossref metadata,
- PubMed identifiers/status,
- correction/retraction signals,
- a separate manual claim-support decision.




In [1]:

# Cell 1 — Colab setup and deterministic output paths

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from difflib import SequenceMatcher
import hashlib
import json
import re
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
OUTDIR = ROOT / "outputs/revision_support/stage6d_1b0_reference_audit_v1"
OUTDIR.mkdir(parents=True, exist_ok=True)

CROSSREF = "https://api.crossref.org"
NCBI = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

HEADERS = {
    "User-Agent": "GES-reference-audit/1.0 (mailto:sbasu23@uis.edu)"
}

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("Output:", OUTDIR)


Mounted at /content/drive
Output: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1b0_reference_audit_v1


In [2]:

# Cell 2 — Freeze the submitted 17-reference bibliography

REFERENCES = [
    {
        "ref": 1,
        "authors": "Landrum MJ, Lee JM, Benson M, et al.",
        "title": "ClinVar: public archive of interpretations of clinically relevant variants",
        "year": 2016,
        "doi_submitted": "10.1093/nar/gkv1222",
        "claim_context": "ClinVar archive structure and clinically interpreted variant records"
    },
    {
        "ref": 2,
        "authors": "Richards S, Aziz N, Bale S, et al.",
        "title": "Standards and guidelines for the interpretation of sequence variants: a joint consensus recommendation of the American College of Medical Genetics and Genomics and the Association for Molecular Pathology",
        "year": 2015,
        "doi_submitted": "10.1038/gim.2015.30",
        "claim_context": "ACMG/AMP sequence-variant interpretation framework"
    },
    {
        "ref": 3,
        "authors": "Harrison SM, Dolinsky JS, Chen W, et al.",
        "title": "Scaling resolution of variant classification differences in ClinVar between clinical laboratories through effortless data sharing",
        "year": 2018,
        "doi_submitted": "10.1002/humu.23643",
        "claim_context": "ClinVar classification differences/disagreement and resolution"
    },
    {
        "ref": 4,
        "authors": "Slavin TP, Van Tongeren LR, Behrendt CE, et al.",
        "title": "Prospective study of cancer genetic variants: variation in rate of reclassification by ancestry",
        "year": 2018,
        "doi_submitted": "10.1093/jnci/djy018",
        "claim_context": "Cancer-variant reclassification dynamics"
    },
    {
        "ref": 5,
        "authors": "Amendola LM, Jarvik GP, Leo MC, et al.",
        "title": "Performance of ACMG-AMP variant-interpretation guidelines among nine laboratories in the Clinical Sequencing Exploratory Research Consortium",
        "year": 2016,
        "doi_submitted": "10.1016/j.ajhg.2016.03.024",
        "claim_context": "Interlaboratory variant-interpretation concordance and guideline performance"
    },
    {
        "ref": 6,
        "authors": "Ghahramani Z",
        "title": "Probabilistic machine learning and artificial intelligence",
        "year": 2015,
        "doi_submitted": "10.1038/nature14541",
        "claim_context": "Probabilistic/model uncertainty"
    },
    {
        "ref": 7,
        "authors": "Angelopoulos AN, Bates S",
        "title": "A gentle introduction to conformal prediction and distribution-free uncertainty quantification",
        "year": 2023,
        "doi_submitted": "10.1561/2200000101",
        "claim_context": "Conformal prediction and uncertainty quantification"
    },
    {
        "ref": 8,
        "authors": "Ratner A, Bach SH, Ehrenberg H, Fries J, Wu S, Re C",
        "title": "Snorkel: rapid training data creation with weak supervision",
        "year": 2017,
        "doi_submitted": "10.14778/3157794.3157797",
        "claim_context": "Weak supervision/programmatic labeling"
    },
    {
        "ref": 9,
        "authors": "Fries JA, Varma P, Chen VS, et al.",
        "title": "Weakly supervised classification of aortic valve malformations using unlabeled cardiac MRI sequences",
        "year": 2019,
        "doi_submitted": "10.1038/s41467-019-11012-3",
        "claim_context": "Biomedical application of weak supervision"
    },
    {
        "ref": 10,
        "authors": "Tonekaboni S, Joshi S, McCradden MD, Goldenberg A",
        "title": "What clinicians want: contextualizing explainable machine learning for clinical end use",
        "year": 2019,
        "doi_submitted": "",
        "stable_url_submitted": "https://proceedings.mlr.press/v106/tonekaboni19a.html",
        "claim_context": "Clinician-centered explainability / trustworthy clinical AI"
    },
    {
        "ref": 11,
        "authors": "Lewis P, Perez E, Piktus A, et al.",
        "title": "Retrieval-augmented generation for knowledge-intensive NLP tasks",
        "year": 2020,
        "doi_submitted": "",
        "stable_url_submitted": "https://arxiv.org/abs/2005.11401",
        "claim_context": "RAG architecture and retrieval-augmented generation"
    },
    {
        "ref": 12,
        "authors": "Zhang G, Xu Z, Jin Q, et al.",
        "title": "Leveraging long context in retrieval augmented language models for medical question answering",
        "year": 2025,
        "doi_submitted": "10.1038/s41746-025-01651-w",
        "claim_context": "Medical RAG and context/retrieval effects"
    },
    {
        "ref": 13,
        "authors": "Gu Y, Tinn R, Cheng H, et al.",
        "title": "Domain-specific language model pretraining for biomedical natural language processing",
        "year": 2021,
        "doi_submitted": "10.1145/3458754",
        "claim_context": "Biomedical-domain language-model pretraining / PubMedBERT"
    },
    {
        "ref": 14,
        "authors": "Johnson J, Douze M, Jegou H",
        "title": "Billion-scale similarity search with GPUs",
        "year": 2019,
        "doi_submitted": "10.1109/TBDATA.2019.2921572",
        "claim_context": "FAISS / dense-vector similarity search"
    },
    {
        "ref": 15,
        "authors": "Pedregosa F, Varoquaux G, Gramfort A, et al.",
        "title": "Scikit-learn: machine learning in Python",
        "year": 2011,
        "doi_submitted": "",
        "stable_url_submitted": "https://jmlr.org/papers/v12/pedregosa11a.html",
        "claim_context": "scikit-learn implementation"
    },
    {
        "ref": 16,
        "authors": "OpenAI",
        "title": "Introducing GPT-4.1 in the API",
        "year": 2025,
        "doi_submitted": "",
        "stable_url_submitted": "https://openai.com/index/gpt-4-1/",
        "claim_context": "Pinned GPT-4.1 model family/API documentation"
    },
    {
        "ref": 17,
        "authors": "Topol EJ",
        "title": "High-performance medicine: the convergence of human and artificial intelligence",
        "year": 2019,
        "doi_submitted": "10.1038/s41591-018-0300-7",
        "claim_context": "Human-centered/trustworthy clinical AI framing"
    },
]

ref_df = pd.DataFrame(REFERENCES).fillna("")
freeze_path = OUTDIR / "stage6d_1b0_submitted_reference_inventory_v1.csv"
ref_df.to_csv(freeze_path, index=False)

print("Frozen reference inventory:", freeze_path)
print("SHA-256:", sha256_file(freeze_path))
display(ref_df)


Frozen reference inventory: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1b0_reference_audit_v1/stage6d_1b0_submitted_reference_inventory_v1.csv
SHA-256: 42fe165888a42b23acc216952c8d37066b84d34faaaeaac1d368f85882937975


,ref,authors,title,year,doi_submitted,claim_context,stable_url_submitted
0,1,"Landrum MJ, Lee JM, Benson M, et al.",ClinVar: public archive of interpretations of ...,2016,10.1093/nar/gkv1222,ClinVar archive structure and clinically inter...,
1,2,"Richards S, Aziz N, Bale S, et al.",Standards and guidelines for the interpretatio...,2015,10.1038/gim.2015.30,ACMG/AMP sequence-variant interpretation frame...,
2,3,"Harrison SM, Dolinsky JS, Chen W, et al.",Scaling resolution of variant classification d...,2018,10.1002/humu.23643,ClinVar classification differences/disagreemen...,
3,4,"Slavin TP, Van Tongeren LR, Behrendt CE, et al.",Prospective study of cancer genetic variants: ...,2018,10.1093/jnci/djy018,Cancer-variant reclassification dynamics,
4,5,"Amendola LM, Jarvik GP, Leo MC, et al.",Performance of ACMG-AMP variant-interpretation...,2016,10.1016/j.ajhg.2016.03.024,Interlaboratory variant-interpretation concord...,
5,6,Ghahramani Z,Probabilistic machine learning and artificial ...,2015,10.1038/nature14541,Probabilistic/model uncertainty,
6,7,"Angelopoulos AN, Bates S",A gentle introduction to conformal prediction ...,2023,10.1561/2200000101,Conformal prediction and uncertainty quantific...,
7,8,"Ratner A, Bach SH, Ehrenberg H, Fries J, Wu S,...",Snorkel: rapid training data creation with wea...,2017,10.14778/3157794.3157797,Weak supervision/programmatic labeling,
8,9,"Fries JA, Varma P, Chen VS, et al.",Weakly supervised classification of aortic val...,2019,10.1038/s41467-019-11012-3,Biomedical application of weak supervision,
9,10,"Tonekaboni S, Joshi S, McCradden MD, Goldenberg A",What clinicians want: contextualizing explaina...,2019,,Clinician-centered explainability / trustworth...,https://proceedings.mlr.press/v106/tonekaboni1...


In [3]:

# Cell 3 — Crossref DOI verification + title-search fallback

def norm_title(s):
    s = re.sub(r"<[^>]+>", " ", str(s))
    s = re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()
    return re.sub(r"\s+", " ", s)

def similarity(a, b):
    return SequenceMatcher(None, norm_title(a), norm_title(b)).ratio()

def crossref_by_doi(doi):
    if not doi:
        return None, None
    url = f"{CROSSREF}/works/{requests.utils.quote(doi, safe='')}"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()["message"], None
        return None, f"HTTP {r.status_code}"
    except Exception as e:
        return None, repr(e)

def crossref_title_search(title, rows=3):
    try:
        r = requests.get(
            f"{CROSSREF}/works",
            headers=HEADERS,
            params={"query.title": title, "rows": rows},
            timeout=30,
        )
        r.raise_for_status()
        return r.json()["message"]["items"]
    except Exception:
        return []

crossref_rows = []

for _, row in ref_df.iterrows():
    doi = row["doi_submitted"].strip()
    meta, err = crossref_by_doi(doi)
    time.sleep(0.12)

    cr_title = ""
    cr_doi = ""
    cr_year = ""
    cr_container = ""
    cr_relation = ""
    cr_match = None
    fallback_doi = ""
    fallback_title = ""
    fallback_score = None

    if meta:
        titles = meta.get("title", [])
        cr_title = titles[0] if titles else ""
        cr_doi = meta.get("DOI", "")
        issued = meta.get("issued", {}).get("date-parts", [[]])
        cr_year = issued[0][0] if issued and issued[0] else ""
        containers = meta.get("container-title", [])
        cr_container = containers[0] if containers else ""
        cr_relation = json.dumps(meta.get("relation", {}), sort_keys=True)
        cr_match = similarity(row["title"], cr_title)
    else:
        candidates = crossref_title_search(row["title"])
        if candidates:
            scored = []
            for item in candidates:
                t = (item.get("title") or [""])[0]
                scored.append((similarity(row["title"], t), item))
            scored.sort(key=lambda x: x[0], reverse=True)
            fallback_score, best = scored[0]
            fallback_doi = best.get("DOI", "")
            fallback_title = (best.get("title") or [""])[0]

    crossref_rows.append({
        "ref": int(row["ref"]),
        "doi_submitted": doi,
        "crossref_lookup_ok": bool(meta),
        "crossref_error": err or "",
        "crossref_doi": cr_doi,
        "crossref_title": cr_title,
        "crossref_title_similarity": cr_match,
        "crossref_year": cr_year,
        "crossref_container": cr_container,
        "crossref_relation": cr_relation,
        "fallback_doi_by_title": fallback_doi,
        "fallback_title": fallback_title,
        "fallback_title_similarity": fallback_score,
    })

crossref_df = pd.DataFrame(crossref_rows)
display(crossref_df)


,ref,doi_submitted,crossref_lookup_ok,crossref_error,crossref_doi,crossref_title,crossref_title_similarity,crossref_year,crossref_container,crossref_relation,fallback_doi_by_title,fallback_title,fallback_title_similarity
0,1,10.1093/nar/gkv1222,True,,10.1093/nar/gkv1222,ClinVar: public archive of interpretations of ...,1.000000,2015,Nucleic Acids Research,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",,,NaN
1,2,10.1038/gim.2015.30,True,,10.1038/gim.2015.30,Standards and guidelines for the interpretatio...,1.000000,2015,Genetics in Medicine,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",,,NaN
2,3,10.1002/humu.23643,True,,10.1002/humu.23643,Scaling resolution of variant classification d...,0.887160,2018,Human Mutation,{},,,NaN
3,4,10.1093/jnci/djy018,True,,10.1093/jnci/djy018,Circulating Tumor Cells in Breast Cancer Patie...,0.358974,2018,JNCI: Journal of the National Cancer Institute,{},,,NaN
4,5,10.1016/j.ajhg.2016.03.024,True,,10.1016/j.ajhg.2016.03.024,Performance of ACMG-AMP Variant-Interpretation...,1.000000,2016,The American Journal of Human Genetics,{},,,NaN
5,6,10.1038/nature14541,True,,10.1038/nature14541,Probabilistic machine learning and artificial ...,1.000000,2015,Nature,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",,,NaN
6,7,10.1561/2200000101,True,,10.1561/2200000101,Conformal Prediction: A Gentle Introduction,0.470588,2023,Foundations and Trends® in Machine Learning,{},,,NaN
7,8,10.14778/3157794.3157797,True,,10.14778/3157794.3157797,Snorkel,0.215385,2017,Proceedings of the VLDB Endowment,{},,,NaN
8,9,10.1038/s41467-019-11012-3,True,,10.1038/s41467-019-11012-3,Weakly supervised classification of aortic val...,1.000000,2019,Nature Communications,"{""has-preprint"": [{""asserted-by"": ""object"", ""i...",,,NaN
9,10,,False,,,,NaN,,,,10.1007/s12559-024-10297-x,Evaluating Explainable Machine Learning Models...,0.653061


In [4]:

# Cell 4 — PubMed DOI/title lookup and correction/retraction relationship screen

def pubmed_esearch(term, retmax=5):
    try:
        r = requests.get(
            f"{NCBI}/esearch.fcgi",
            params={"db": "pubmed", "term": term, "retmode": "json", "retmax": retmax},
            timeout=30,
        )
        r.raise_for_status()
        return r.json()["esearchresult"]["idlist"]
    except Exception:
        return []

def pubmed_fetch_xml(pmid):
    try:
        r = requests.get(
            f"{NCBI}/efetch.fcgi",
            params={"db": "pubmed", "id": pmid, "retmode": "xml"},
            timeout=30,
        )
        r.raise_for_status()
        return ET.fromstring(r.text)
    except Exception:
        return None

def text_of(node, path):
    x = node.find(path)
    return "" if x is None else "".join(x.itertext()).strip()

def parse_pubmed(pmid, root):
    article = root.find(".//PubmedArticle")
    if article is None:
        return {}
    title_node = article.find(".//ArticleTitle")
    title = "" if title_node is None else "".join(title_node.itertext()).strip()

    doi = ""
    for aid in article.findall(".//ArticleId"):
        if aid.attrib.get("IdType") == "doi":
            doi = (aid.text or "").strip()

    pubtypes = [
        "".join(x.itertext()).strip()
        for x in article.findall(".//PublicationType")
    ]

    cc = []
    for x in article.findall(".//CommentsCorrections"):
        cc.append({
            "RefType": x.attrib.get("RefType", ""),
            "PMID": text_of(x, "PMID"),
            "RefSource": text_of(x, "RefSource"),
            "Note": text_of(x, "Note"),
        })

    return {
        "pmid": pmid,
        "pubmed_title": title,
        "pubmed_doi": doi,
        "publication_types": "; ".join(pubtypes),
        "comments_corrections": json.dumps(cc, ensure_ascii=False),
    }

pubmed_rows = []

for _, row in ref_df.iterrows():
    ref = int(row["ref"])
    submitted_doi = row["doi_submitted"].strip()

    crrow = crossref_df.loc[crossref_df["ref"] == ref].iloc[0]
    suggested_doi = (
        crrow["crossref_doi"]
        if crrow["crossref_lookup_ok"]
        else crrow["fallback_doi_by_title"]
    )

    candidate_doi = submitted_doi or str(suggested_doi or "").strip()

    pmids = []
    search_method = ""
    if candidate_doi:
        pmids = pubmed_esearch(f'"{candidate_doi}"[AID]')
        search_method = "doi"
        time.sleep(0.12)

    if not pmids:
        safe_title = row["title"].replace('"', "")
        pmids = pubmed_esearch(f'"{safe_title}"[Title]')
        search_method = "title"
        time.sleep(0.12)

    rec = {
        "ref": ref,
        "pubmed_found": False,
        "pmid": "",
        "pubmed_search_method": search_method,
        "pubmed_title": "",
        "pubmed_title_similarity": None,
        "pubmed_doi": "",
        "publication_types": "",
        "comments_corrections": "",
        "retraction_or_correction_flag": False,
    }

    if pmids:
        root = pubmed_fetch_xml(pmids[0])
        time.sleep(0.12)
        parsed = parse_pubmed(pmids[0], root) if root is not None else {}
        if parsed:
            rec.update(parsed)
            rec["pubmed_found"] = True
            rec["pubmed_search_method"] = search_method
            rec["pubmed_title_similarity"] = similarity(row["title"], parsed["pubmed_title"])
            cc_lower = parsed["comments_corrections"].lower()
            pt_lower = parsed["publication_types"].lower()
            trigger_words = [
                "retract", "retraction", "erratum", "correct", "expressionofconcern"
            ]
            rec["retraction_or_correction_flag"] = (
                any(w in cc_lower for w in trigger_words)
                or "retracted publication" in pt_lower
            )

    pubmed_rows.append(rec)

pubmed_df = pd.DataFrame(pubmed_rows)
display(pubmed_df)


,ref,pubmed_found,pmid,pubmed_search_method,pubmed_title,pubmed_title_similarity,pubmed_doi,publication_types,comments_corrections,retraction_or_correction_flag
0,1,True,26582918,doi,ClinVar: public archive of interpretations of ...,1.000000,10.1093/nar/gkv1222,"Journal Article; Research Support, N.I.H., Int...",[],False
1,2,True,25741868,doi,Standards and guidelines for the interpretatio...,1.000000,10.1038/gim.2015.30,Consensus Statement; Journal Article; Research...,"[{""RefType"": ""CommentIn"", ""PMID"": ""25854183"", ...",False
2,3,True,30311378,doi,Scaling resolution of variant classification d...,0.887160,10.1038/gim.2017.60,"Journal Article; Research Support, N.I.H., Ext...",[],False
3,4,True,29659933,doi,Circulating Tumor Cells in Breast Cancer Patie...,0.358974,10.1093/jnci/djy018,Journal Article; Meta-Analysis; Research Suppo...,"[{""RefType"": ""CommentIn"", ""PMID"": ""29659941"", ...",False
4,5,True,27181684,doi,Performance of ACMG-AMP Variant-Interpretation...,1.000000,10.1016/j.ajhg.2016.03.024,"Journal Article; Research Support, N.I.H., Ext...","[{""RefType"": ""ErratumIn"", ""PMID"": ""27392081"", ...",True
5,6,True,26017444,doi,Probabilistic machine learning and artificial ...,1.000000,10.1038/nature14541,"Journal Article; Research Support, Non-U.S. Go...",[],False
6,7,False,,title,,NaN,,,,False
7,8,False,,title,,NaN,,,,False
8,9,True,31308376,doi,Weakly supervised classification of aortic val...,1.000000,10.1038/s41746-017-0015-z,"Journal Article; Research Support, N.I.H., Ext...",[],False
9,10,False,,title,,NaN,,,,False


In [5]:

# Cell 5 — Consolidated authenticity audit and mismatch flags

audit = (
    ref_df
    .merge(crossref_df, on=["ref", "doi_submitted"], how="left")
    .merge(pubmed_df, on="ref", how="left")
)

def doi_norm(x):
    return str(x or "").lower().strip().replace("https://doi.org/", "").replace("http://doi.org/", "")

audit["submitted_vs_crossref_doi_match"] = audit.apply(
    lambda r: (
        True if not doi_norm(r["doi_submitted"]) or not doi_norm(r["crossref_doi"])
        else doi_norm(r["doi_submitted"]) == doi_norm(r["crossref_doi"])
    ),
    axis=1
)

audit["title_match_pass"] = audit.apply(
    lambda r: (
        (pd.notna(r["crossref_title_similarity"]) and r["crossref_title_similarity"] >= 0.90)
        or
        (pd.notna(r["pubmed_title_similarity"]) and r["pubmed_title_similarity"] >= 0.90)
    ),
    axis=1
)

audit["metadata_status"] = "REVIEW"
audit.loc[
    (
        audit["title_match_pass"]
        & (
            audit["crossref_lookup_ok"]
            | audit["pubmed_found"]
            | (audit["stable_url_submitted"].astype(str).str.len() > 0)
        )
    ),
    "metadata_status"
] = "TRACEABLE"

# A DOI that resolves to a title inconsistent with the submitted citation is high-priority.
audit["doi_title_mismatch_flag"] = (
    audit["doi_submitted"].astype(str).str.len().gt(0)
    & audit["crossref_lookup_ok"].fillna(False)
    & audit["crossref_title_similarity"].fillna(0).lt(0.90)
)

# If submitted DOI fails but a high-similarity title search finds another DOI, flag it.
audit["candidate_corrected_doi_flag"] = (
    (~audit["crossref_lookup_ok"].fillna(False))
    & audit["fallback_title_similarity"].fillna(0).ge(0.90)
    & audit["fallback_doi_by_title"].astype(str).str.len().gt(0)
)

cols = [
    "ref", "authors", "title", "year", "doi_submitted",
    "metadata_status", "doi_title_mismatch_flag", "candidate_corrected_doi_flag",
    "crossref_title", "crossref_title_similarity", "fallback_doi_by_title",
    "fallback_title_similarity", "pmid", "pubmed_doi",
    "retraction_or_correction_flag", "comments_corrections",
]
display(audit[cols])


,ref,authors,title,year,doi_submitted,metadata_status,doi_title_mismatch_flag,candidate_corrected_doi_flag,crossref_title,crossref_title_similarity,fallback_doi_by_title,fallback_title_similarity,pmid,pubmed_doi,retraction_or_correction_flag,comments_corrections
0,1,"Landrum MJ, Lee JM, Benson M, et al.",ClinVar: public archive of interpretations of ...,2016,10.1093/nar/gkv1222,TRACEABLE,False,False,ClinVar: public archive of interpretations of ...,1.000000,,NaN,26582918,10.1093/nar/gkv1222,False,[]
1,2,"Richards S, Aziz N, Bale S, et al.",Standards and guidelines for the interpretatio...,2015,10.1038/gim.2015.30,TRACEABLE,False,False,Standards and guidelines for the interpretatio...,1.000000,,NaN,25741868,10.1038/gim.2015.30,False,"[{""RefType"": ""CommentIn"", ""PMID"": ""25854183"", ..."
2,3,"Harrison SM, Dolinsky JS, Chen W, et al.",Scaling resolution of variant classification d...,2018,10.1002/humu.23643,REVIEW,True,False,Scaling resolution of variant classification d...,0.887160,,NaN,30311378,10.1038/gim.2017.60,False,[]
3,4,"Slavin TP, Van Tongeren LR, Behrendt CE, et al.",Prospective study of cancer genetic variants: ...,2018,10.1093/jnci/djy018,REVIEW,True,False,Circulating Tumor Cells in Breast Cancer Patie...,0.358974,,NaN,29659933,10.1093/jnci/djy018,False,"[{""RefType"": ""CommentIn"", ""PMID"": ""29659941"", ..."
4,5,"Amendola LM, Jarvik GP, Leo MC, et al.",Performance of ACMG-AMP variant-interpretation...,2016,10.1016/j.ajhg.2016.03.024,TRACEABLE,False,False,Performance of ACMG-AMP Variant-Interpretation...,1.000000,,NaN,27181684,10.1016/j.ajhg.2016.03.024,True,"[{""RefType"": ""ErratumIn"", ""PMID"": ""27392081"", ..."
5,6,Ghahramani Z,Probabilistic machine learning and artificial ...,2015,10.1038/nature14541,TRACEABLE,False,False,Probabilistic machine learning and artificial ...,1.000000,,NaN,26017444,10.1038/nature14541,False,[]
6,7,"Angelopoulos AN, Bates S",A gentle introduction to conformal prediction ...,2023,10.1561/2200000101,REVIEW,True,False,Conformal Prediction: A Gentle Introduction,0.470588,,NaN,,,False,
7,8,"Ratner A, Bach SH, Ehrenberg H, Fries J, Wu S,...",Snorkel: rapid training data creation with wea...,2017,10.14778/3157794.3157797,REVIEW,True,False,Snorkel,0.215385,,NaN,,,False,
8,9,"Fries JA, Varma P, Chen VS, et al.",Weakly supervised classification of aortic val...,2019,10.1038/s41467-019-11012-3,TRACEABLE,False,False,Weakly supervised classification of aortic val...,1.000000,,NaN,31308376,10.1038/s41746-017-0015-z,False,[]
9,10,"Tonekaboni S, Joshi S, McCradden MD, Goldenberg A",What clinicians want: contextualizing explaina...,2019,,REVIEW,False,False,,NaN,10.1007/s12559-024-10297-x,0.653061,,,False,


In [6]:

# Cell 6 — Materialize the manual claim-support verification sheet

claim_sheet = audit[
    [
        "ref", "authors", "title", "year", "doi_submitted",
        "fallback_doi_by_title", "pmid", "claim_context",
        "metadata_status", "doi_title_mismatch_flag",
        "candidate_corrected_doi_flag", "retraction_or_correction_flag",
    ]
].copy()

claim_sheet["final_identifier_to_use"] = ""
claim_sheet["source_opened_manually"] = ""
claim_sheet["claim_supported_by_source"] = ""
claim_sheet["correction_retraction_checked_manually"] = ""
claim_sheet["retain_replace_remove"] = ""
claim_sheet["manual_notes"] = ""

claim_path = OUTDIR / "stage6d_1b0_manual_claim_support_verification_sheet_v1.csv"
claim_sheet.to_csv(claim_path, index=False)

print("Manual verification sheet:", claim_path)
display(claim_sheet)


Manual verification sheet: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1b0_reference_audit_v1/stage6d_1b0_manual_claim_support_verification_sheet_v1.csv


,ref,authors,title,year,doi_submitted,fallback_doi_by_title,pmid,claim_context,metadata_status,doi_title_mismatch_flag,candidate_corrected_doi_flag,retraction_or_correction_flag,final_identifier_to_use,source_opened_manually,claim_supported_by_source,correction_retraction_checked_manually,retain_replace_remove,manual_notes
0,1,"Landrum MJ, Lee JM, Benson M, et al.",ClinVar: public archive of interpretations of ...,2016,10.1093/nar/gkv1222,,26582918,ClinVar archive structure and clinically inter...,TRACEABLE,False,False,False,,,,,,
1,2,"Richards S, Aziz N, Bale S, et al.",Standards and guidelines for the interpretatio...,2015,10.1038/gim.2015.30,,25741868,ACMG/AMP sequence-variant interpretation frame...,TRACEABLE,False,False,False,,,,,,
2,3,"Harrison SM, Dolinsky JS, Chen W, et al.",Scaling resolution of variant classification d...,2018,10.1002/humu.23643,,30311378,ClinVar classification differences/disagreemen...,REVIEW,True,False,False,,,,,,
3,4,"Slavin TP, Van Tongeren LR, Behrendt CE, et al.",Prospective study of cancer genetic variants: ...,2018,10.1093/jnci/djy018,,29659933,Cancer-variant reclassification dynamics,REVIEW,True,False,False,,,,,,
4,5,"Amendola LM, Jarvik GP, Leo MC, et al.",Performance of ACMG-AMP variant-interpretation...,2016,10.1016/j.ajhg.2016.03.024,,27181684,Interlaboratory variant-interpretation concord...,TRACEABLE,False,False,True,,,,,,
5,6,Ghahramani Z,Probabilistic machine learning and artificial ...,2015,10.1038/nature14541,,26017444,Probabilistic/model uncertainty,TRACEABLE,False,False,False,,,,,,
6,7,"Angelopoulos AN, Bates S",A gentle introduction to conformal prediction ...,2023,10.1561/2200000101,,,Conformal prediction and uncertainty quantific...,REVIEW,True,False,False,,,,,,
7,8,"Ratner A, Bach SH, Ehrenberg H, Fries J, Wu S,...",Snorkel: rapid training data creation with wea...,2017,10.14778/3157794.3157797,,,Weak supervision/programmatic labeling,REVIEW,True,False,False,,,,,,
8,9,"Fries JA, Varma P, Chen VS, et al.",Weakly supervised classification of aortic val...,2019,10.1038/s41467-019-11012-3,,31308376,Biomedical application of weak supervision,TRACEABLE,False,False,False,,,,,,
9,10,"Tonekaboni S, Joshi S, McCradden MD, Goldenberg A",What clinicians want: contextualizing explaina...,2019,,10.1007/s12559-024-10297-x,,Clinician-centered explainability / trustworth...,REVIEW,False,False,False,,,,,,


In [7]:

# Cell 7 — Save machine audit outputs and compute explicit review queue

crossref_path = OUTDIR / "stage6d_1b0_crossref_results_v1.csv"
pubmed_path = OUTDIR / "stage6d_1b0_pubmed_results_v1.csv"
audit_path = OUTDIR / "stage6d_1b0_consolidated_reference_audit_v1.csv"

crossref_df.to_csv(crossref_path, index=False)
pubmed_df.to_csv(pubmed_path, index=False)
audit.to_csv(audit_path, index=False)

review_queue = audit[
    (audit["metadata_status"] != "TRACEABLE")
    | audit["doi_title_mismatch_flag"]
    | audit["candidate_corrected_doi_flag"]
    | audit["retraction_or_correction_flag"]
].copy()

queue_path = OUTDIR / "stage6d_1b0_reference_review_queue_v1.csv"
review_queue.to_csv(queue_path, index=False)

print("Traceable:", int((audit["metadata_status"] == "TRACEABLE").sum()), "/", len(audit))
print("DOI/title mismatch flags:", int(audit["doi_title_mismatch_flag"].sum()))
print("Candidate corrected DOI flags:", int(audit["candidate_corrected_doi_flag"].sum()))
print("Correction/retraction flags requiring manual inspection:",
      int(audit["retraction_or_correction_flag"].sum()))
print("Review queue:", queue_path)

display(review_queue[cols] if len(review_queue) else pd.DataFrame({"status": ["No machine flags"]}))


Traceable: 9 / 17
DOI/title mismatch flags: 4
Candidate corrected DOI flags: 0
Correction/retraction flags requiring manual inspection: 1
Review queue: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1b0_reference_audit_v1/stage6d_1b0_reference_review_queue_v1.csv


,ref,authors,title,year,doi_submitted,metadata_status,doi_title_mismatch_flag,candidate_corrected_doi_flag,crossref_title,crossref_title_similarity,fallback_doi_by_title,fallback_title_similarity,pmid,pubmed_doi,retraction_or_correction_flag,comments_corrections
2,3,"Harrison SM, Dolinsky JS, Chen W, et al.",Scaling resolution of variant classification d...,2018,10.1002/humu.23643,REVIEW,True,False,Scaling resolution of variant classification d...,0.887160,,NaN,30311378,10.1038/gim.2017.60,False,[]
3,4,"Slavin TP, Van Tongeren LR, Behrendt CE, et al.",Prospective study of cancer genetic variants: ...,2018,10.1093/jnci/djy018,REVIEW,True,False,Circulating Tumor Cells in Breast Cancer Patie...,0.358974,,NaN,29659933,10.1093/jnci/djy018,False,"[{""RefType"": ""CommentIn"", ""PMID"": ""29659941"", ..."
4,5,"Amendola LM, Jarvik GP, Leo MC, et al.",Performance of ACMG-AMP variant-interpretation...,2016,10.1016/j.ajhg.2016.03.024,TRACEABLE,False,False,Performance of ACMG-AMP Variant-Interpretation...,1.000000,,NaN,27181684,10.1016/j.ajhg.2016.03.024,True,"[{""RefType"": ""ErratumIn"", ""PMID"": ""27392081"", ..."
6,7,"Angelopoulos AN, Bates S",A gentle introduction to conformal prediction ...,2023,10.1561/2200000101,REVIEW,True,False,Conformal Prediction: A Gentle Introduction,0.470588,,NaN,,,False,
7,8,"Ratner A, Bach SH, Ehrenberg H, Fries J, Wu S,...",Snorkel: rapid training data creation with wea...,2017,10.14778/3157794.3157797,REVIEW,True,False,Snorkel,0.215385,,NaN,,,False,
9,10,"Tonekaboni S, Joshi S, McCradden MD, Goldenberg A",What clinicians want: contextualizing explaina...,2019,,REVIEW,False,False,,NaN,10.1007/s12559-024-10297-x,0.653061,,,False,
10,11,"Lewis P, Perez E, Piktus A, et al.",Retrieval-augmented generation for knowledge-i...,2020,,REVIEW,False,False,,NaN,10.18653/v1/2022.naacl-main.162,0.806202,,,False,
14,15,"Pedregosa F, Varoquaux G, Gramfort A, et al.",Scikit-learn: machine learning in Python,2011,,REVIEW,False,False,,NaN,10.1007/978-1-4842-9532-8_8,0.575342,,,False,
15,16,OpenAI,Introducing GPT-4.1 in the API,2025,,REVIEW,False,False,,NaN,10.1007/978-1-4842-5184-3_1,0.775510,,,False,


In [8]:

# Cell 8 — Special explicit check for Reference 4 (known high-priority item)

r4 = audit.loc[audit["ref"] == 4].iloc[0]

print("REFERENCE 4")
print("Submitted title:", r4["title"])
print("Submitted DOI:", r4["doi_submitted"])
print("Crossref title from submitted DOI:", r4["crossref_title"])
print("Submitted-DOI title similarity:", r4["crossref_title_similarity"])
print("Fallback DOI found by title:", r4["fallback_doi_by_title"])
print("Fallback title similarity:", r4["fallback_title_similarity"])
print("PubMed PMID:", r4["pmid"])
print("PubMed DOI:", r4["pubmed_doi"])

if (
    r4["candidate_corrected_doi_flag"]
    or r4["doi_title_mismatch_flag"]
    or doi_norm(r4["pubmed_doi"]) != doi_norm(r4["doi_submitted"])
):
    print("\nACTION REQUIRED — Reference 4 identifier must be manually verified before final manuscript.")
else:
    print("\nNo machine-level identifier discrepancy detected; still perform manual source verification.")


REFERENCE 4
Submitted title: Prospective study of cancer genetic variants: variation in rate of reclassification by ancestry
Submitted DOI: 10.1093/jnci/djy018
Crossref title from submitted DOI: Circulating Tumor Cells in Breast Cancer Patients Treated by Neoadjuvant Chemotherapy: A Meta-analysis
Submitted-DOI title similarity: 0.358974358974359
Fallback DOI found by title: 
Fallback title similarity: nan
PubMed PMID: 29659933
PubMed DOI: 10.1093/jnci/djy018

ACTION REQUIRED — Reference 4 identifier must be manually verified before final manuscript.


In [9]:

# Cell 9 — Reproducibility manifest and immutable hashes

artifacts = [
    freeze_path,
    crossref_path,
    pubmed_path,
    audit_path,
    queue_path,
    claim_path,
]

manifest_rows = []
for p in artifacts:
    manifest_rows.append({
        "artifact": str(p),
        "bytes": Path(p).stat().st_size,
        "sha256": sha256_file(Path(p)),
    })

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUTDIR / "stage6d_1b0_reference_audit_manifest_v1.csv"
manifest.to_csv(manifest_path, index=False)

for p in artifacts + [manifest_path]:
    p = Path(p)
    p.with_name(p.name + ".sha256").write_text(
        f"{sha256_file(p)}  {p.name}\n",
        encoding="utf-8",
    )

print("Manifest:", manifest_path)
display(manifest)


Manifest: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1b0_reference_audit_v1/stage6d_1b0_reference_audit_manifest_v1.csv


,artifact,bytes,sha256
0,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,3590,42fe165888a42b23acc216952c8d37066b84d34faaaeaa...
1,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,3701,aa1f56f7b0b0d393914a62297b77e2e481ba3bcfee54d4...
2,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,4144,3972fb60f8fbbeedffe36e14764cc15adfb786124fbe20...
3,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,11673,bbf3e9181e4b9d69c68cfea4be9c98919893d120943f70...
4,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,5287,7b653eaddee4765570d2a644d683c06e083041d66b0696...
5,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,4420,5898e085580ddd76b15a6cf52d88acfdc7d360bd23a509...


In [10]:

# Cell 10 — Final reviewer-facing status

traceable = int((audit["metadata_status"] == "TRACEABLE").sum())
mismatch = int(audit["doi_title_mismatch_flag"].sum())
corrected = int(audit["candidate_corrected_doi_flag"].sum())
status_flags = int(audit["retraction_or_correction_flag"].sum())

print("FINAL MACHINE AUDIT")
print("-------------------")
print(f"References inventoried: {len(audit)}")
print(f"Machine-traceable references: {traceable}")
print(f"Submitted DOI/title mismatch flags: {mismatch}")
print(f"Candidate alternate/corrected DOI flags: {corrected}")
print(f"Correction/retraction relationship flags: {status_flags}")

print("""
NEXT REQUIRED HUMAN STEP:
Open stage6d_1b0_manual_claim_support_verification_sheet_v1.csv and complete,
for every retained reference:
  1) final identifier,
  2) source manually opened,
  3) cited manuscript claim actually supported,
  4) correction/retraction status manually checked on PubMed/Crossmark/publisher,
  5) retain / replace / remove decision.

Do not tell the Editorial Board that all references are verified until every
retained reference has completed manual confirmation.
""")


FINAL MACHINE AUDIT
-------------------
References inventoried: 17
Machine-traceable references: 9
Submitted DOI/title mismatch flags: 4
Candidate alternate/corrected DOI flags: 0
Correction/retraction relationship flags: 1

NEXT REQUIRED HUMAN STEP:
Open stage6d_1b0_manual_claim_support_verification_sheet_v1.csv and complete,
for every retained reference:
  1) final identifier,
  2) source manually opened,
  3) cited manuscript claim actually supported,
  4) correction/retraction status manually checked on PubMed/Crossmark/publisher,
  5) retain / replace / remove decision.

Do not tell the Editorial Board that all references are verified until every
retained reference has completed manual confirmation.




## After this notebook

Once the manual claim-support sheet is complete, the next revision-support step is:

**Stage 6D-1C0 — Final Springer Revision Evidence Package Freeze**

That package should freeze the exact:
- ClinVar release identifiers and dates,
- variant/RCV counts,
- code repository commit,
- software environment,
- temporal-validation table,
- data-quality perturbation table/figure,
- final reference audit,
- declarations,
- data/code availability text,
- point-by-point response evidence map.

Do **not** proceed to the final response letter before the reference audit is complete.
